#### Imports

In [ ]:
# Imports
import os
import sys
import json
import torch
from tqdm import tqdm
from PIL import Image
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from transformers import RTDetrV2ForObjectDetection, RTDetrImageProcessor

#### Path Configurations

In [2]:
# Get the absolute path to the 'project' directory
project_root = os.path.abspath(os.path.join('..', '..'))

# Add it to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

# Path Configuration
data_directory = os.path.join(project_root , "data" , "detection")
dataset_name = "coco_football_players_detection_v11"
dataset_path = os.path.join(data_directory , dataset_name)

# Model Name and Weights
checkpoint_num = 6109
base_model_name = "PekingU/rtdetr_v2_r101vd"
model_name = "28-04-2026_21-08_rtdetr_v2_r101vd"
model_directory = os.path.join(project_root , "models" , "detection", model_name)
full_model_weights_path = os.path.join(model_directory, f"checkpoint-{checkpoint_num}")

#### Model Setup

In [3]:
# Detection Model
categories = ['ball', 'goalkeeper', 'player', 'referee']
id2label = {index: x for index, x in enumerate(categories, start=0)}
label2id = {v: k for k, v in id2label.items()}


# Image Processor
image_processor = RTDetrImageProcessor.from_pretrained(
    base_model_name,
    do_resize=True,
    size={"width": 1280, "height": 1280},
    use_Fast=True,
)

# Detection Model
detection_model = RTDetrV2ForObjectDetection.from_pretrained(
    full_model_weights_path,
    num_labels=len(categories),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # needed when num_labels differs from the checkpoint  
    local_files_only=True,  
)

Loading weights: 100%|██████████| 1025/1025 [00:00<00:00, 2889.90it/s]


#### Loading Dataset

In [4]:
test_dataset = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset_path}/test",
    annotations_path=f"{dataset_path}/test/_annotations.coco.json",
)

#### Model Metrics

In [5]:
# Gathering Model Statistics
targets = []
predictions = []


for path, image, annotations in tqdm(test_dataset):
    image = Image.open(path)

    # Run inference
    inputs = image_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = detection_model(**inputs)

    # Convert outputs to sv.Detections
    # post_process_object_detection applies the softmax + threshold internally
    target_sizes = torch.tensor([image.size[::-1]])  # (height, width)
    results = image_processor.post_process_object_detection(
        outputs,
        threshold=0.0,   # keep everything, let MAP handle it
        target_sizes=target_sizes
    )[0]

    detections = sv.Detections(
        xyxy=results["boxes"].cpu().numpy(),
        confidence=results["scores"].cpu().numpy(),
        class_id=results["labels"].cpu().numpy(),
    )

    targets.append(annotations)
    predictions.append(detections)

100%|██████████| 25/25 [01:09<00:00,  2.80s/it]


#### Gathering Additional Training Configs and Stats

In [6]:
# Paths for training arguments and results
model_training_args_path = os.path.join(full_model_weights_path, 'trainer_state.json')

# Gathering training configuration
with open(model_training_args_path, "r") as f:
    training_config = json.load(f)

#### Per-class and Overall Stats

In [7]:
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()

class_names = test_dataset.classes

per_class_metrics = {}
for i, name in enumerate(class_names):
    per_class_metrics[name] = {
        "mAP50-95": round(float(map_result.ap_per_class[i].mean()), 4),
        "mAP50":    round(float(map_result.ap_per_class[i, 0]), 4),
    }

overall_metrics = {
    "mAP50-95": round(float(map_result.map50_95), 4),
    "mAP50":    round(float(map_result.map50), 4),
}

#### Full Metrics

In [8]:
# Full Metrics
metrics_to_save = {
    'model_name': model_name,
    'pretrained_base_model': base_model_name,
    'evaluation_dataset': dataset_name,
    'imgsz': 1280,
    'intended_epochs_trained': training_config['num_train_epochs'],
    'epochs_trained': training_config['epoch'],
    'overall_metrics': overall_metrics,
    'per_class_overall_metrics':per_class_metrics,
    'total_flos': training_config['total_flos'],
    'ending_training_metrics' : {
        "mAP50":     training_config['log_history'][-1]['eval_map_50'],
        "mAP75":     training_config['log_history'][-1]['eval_map_75'],
        "mAP50-95":     training_config['log_history'][-1]['eval_map'],
        },
    'training_args': training_config
    }

#### Saving JSON Metrics

In [9]:
full_model_eval_results_path = os.path.join(model_directory, f'{model_name}_{dataset_name}_results.json')
with open(full_model_eval_results_path, "w") as f:
    json.dump(metrics_to_save, f, indent=4)